# Exact subtraction with native FlagQuantum

Preserve both original tutorial examples and their order. Use coherent quantum values, local CPU execution and exact named results; seeded local samples replace the upstream cloud sampler. No hardware result is claimed.


In [ ]:
from collections import Counter
from fractions import Fraction
from random import Random
from typing import Annotated

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QNum, allocate, hadamard_transform, qfunc, synthesize


## Subtract words

The original values and input domains are preserved. Any signed/fractional values below use exact numeric readout rather than manual bit-string decoding.


In [ ]:
@qfunc
def subtract_words(a: Output[QNum], b: Output[QNum], res: Output[QNum]) -> None:
    a |= 4
    b |= 5
    res |= a - b


In [ ]:
subtract_words_program = synthesize(subtract_words)
subtract_words_result = fq.run(subtract_words_program)
subtract_words_probabilities = dict(subtract_words_program.numeric_probabilities(subtract_words_result, registers=('a', 'b', 'res')))
expected = {(4, 5, -1): 1.0}
assert subtract_words_probabilities.keys() == expected.keys()
assert all(abs(subtract_words_probabilities[key]-value) < 1e-12 for key,value in expected.items())
subtract_words_counts = Counter(Random(17).choices(list(subtract_words_probabilities), weights=list(subtract_words_probabilities.values()), k=1000))
assert sum(subtract_words_counts.values()) == 1000
print("Probabilities:", subtract_words_probabilities)
print("Counts:", dict(subtract_words_counts))
print("Resources:", subtract_words_program.report["resources"])


## Subtract scaled

The original values and input domains are preserved. Any signed/fractional values below use exact numeric readout rather than manual bit-string decoding.


In [ ]:
@qfunc
def subtract_scaled(a: Output[QNum], res: Output[QNum]) -> None:
    a |= 4
    res |= a - 3.5


In [ ]:
subtract_scaled_program = synthesize(subtract_scaled)
subtract_scaled_result = fq.run(subtract_scaled_program)
subtract_scaled_probabilities = dict(subtract_scaled_program.numeric_probabilities(subtract_scaled_result, registers=('a', 'res')))
expected = {(4, Fraction(1, 2)): 1.0}
assert subtract_scaled_probabilities.keys() == expected.keys()
assert all(abs(subtract_scaled_probabilities[key]-value) < 1e-12 for key,value in expected.items())
subtract_scaled_counts = Counter(Random(17).choices(list(subtract_scaled_probabilities), weights=list(subtract_scaled_probabilities.values()), k=1000))
assert sum(subtract_scaled_counts.values()) == 1000
print("Probabilities:", subtract_scaled_probabilities)
print("Counts:", dict(subtract_scaled_counts))
print("Resources:", subtract_scaled_program.report["resources"])


## What is checked

These examples keep their original inputs, including exact binary coefficient 3.5 where present. Independent Circuit acceptance must additionally check the complete complex state and work cleanup; probabilities and samples alone do not certify phase.
